# 👥 Multi-User Conversational RAG System

Turn the conversational RAG chain into a chatbot that many users can talk to at once: every user (session) gets their
own chat history, stored in a SQL database and added to the chain automatically.

**Reference:** [RAG Systems Essentials (Analytics Vidhya)](https://courses.analyticsvidhya.com/courses/take/rag-systems-essentials/lessons/60148017-hands-on-deep-dive-into-rag-evaluation-metrics-generator-metrics-i)

## Learning Objectives
In this notebook, you will learn:
1. **Build the knowledge base** - the same Wikipedia subset and Chroma retriever as the previous notebook
2. **History-aware retriever** - rewrite follow-up questions into standalone questions before retrieval
3. **Per-session history in SQL** - `SQLChatMessageHistory` keyed by `session_id`, plus a last-K message window
4. **Wire history in automatically** - `RunnableWithMessageHistory` loads and saves each session's messages
5. **Serve several users** - interleave two users' conversations and check that their histories stay separate

## Where this fits
`03_conversational_rag.ipynb` kept one chat history in a Python list that the caller extended by hand. This notebook
moves the history into SQLite, one history per session, so the chain itself remembers each user.

## Prerequisites
- `03_conversational_rag.ipynb` (history-aware retriever, `create_retrieval_chain`) - background reading
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (OpenAI embeddings and `gpt-4o-mini`)
- Internet access: Part 1 downloads the Simple English Wikipedia dump from sbert.net on its first run

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from this project's `.env`, then create the chat model used for question rewriting and answers.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: gpt-4o-mini for question rewriting and answers
# ============================================================================
from aied_helpers import get_openai_llm

# temperature=0 is the factory default, as in the original lesson
llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📚 Part 1: Build the Wikipedia Knowledge Base

The same knowledge base as `03_conversational_rag.ipynb`: the Simple English Wikipedia dump, first paragraphs only,
filtered to passages that mention *fish*, *india* or *cheetah*, chunked and indexed in Chroma. If that notebook already
built `./wiki_db` in this folder, the build cell below reuses it.

In [ ]:
# ============================================================================
# DOWNLOAD: Simple English Wikipedia dump (2020-11-01) from sbert.net
# ============================================================================
import gzip
import json
import requests
from tqdm import tqdm
import sys
import os

# Download a file from a URL
def http_get(url, path) -> None:
    """
    Downloads a URL to a given path on disc
    """
    if os.path.dirname(path) != "":
        os.makedirs(os.path.dirname(path), exist_ok=True)

    req = requests.get(url, stream=True)
    if req.status_code != 200:
        print("Exception when trying to download {}. Response {}".format(url, req.status_code), file=sys.stderr)
        req.raise_for_status()
        return

    download_filepath = path + "_part"
    with open(download_filepath, "wb") as file_binary:
        content_length = req.headers.get("Content-Length")
        total = int(content_length) if content_length is not None else None
        progress = tqdm(unit="B", total=total, unit_scale=True)
        for chunk in req.iter_content(chunk_size=1024):
            if chunk:  # filter out keep-alive new chunks
                progress.update(len(chunk))
                file_binary.write(chunk)

    os.rename(download_filepath, path)
    progress.close()


wikipedia_filepath = 'simplewiki-2020-11-01.jsonl.gz'

# download once; later runs (and the other conversational notebook) reuse the local file
if not os.path.exists(wikipedia_filepath):
    http_get('http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz', wikipedia_filepath)
print('✅ Wikipedia dump ready:', wikipedia_filepath)

In [ ]:
# ============================================================================
# PARSE: Keep the first paragraph of every article
# ============================================================================
import gzip
import json

wikipedia_filepath = 'simplewiki-2020-11-01.jsonl.gz'

passages = []
with gzip.open(wikipedia_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        data = json.loads(line.strip())
        #Only add the first paragraph
        passages.append(data['paragraphs'][0])

print("Passages:", len(passages))

In [ ]:
# We subset our data so we only use a subset of wikipedia to run things faster
passages = [passage for passage in passages for x in ['fish', 'india', 'cheetah']
              if x in passage.lower().split()]

In [ ]:
len(passages)

In [ ]:
passages[0]

### Generate LLM Embeddings and store them in Chroma Vector DB

**Chroma Vector DB** is a versatile, open-source vector database designed for managing and querying vector embeddings. It is easy to set up and integrates well with various AI tools and algorithms. Chroma is particularly useful for applications that require rapid and precise retrieval of content represented as embeddings—efficient data formats for text, images, and soon, audio and video.

**Key Features:**
- **Integration with AI Tools:** Chroma supports embedding functions from leading providers like OpenAI, Google, and Hugging Face, allowing for flexible and powerful data handling.
- **Ease of Use:** The database provides default embedding functions, or users can integrate external APIs to generate embeddings.
- **Efficient Querying:** Users can create collections to store embeddings, documents, and metadata. These can be queried to retrieve the most similar items, making information retrieval quick and effective.
- **Flexible API:** Chroma offers a straightforward API that supports both standard operations and custom embedding functions.

For more detailed information, visit the official Chroma documentation [here](https://docs.trychroma.com).

In [ ]:
passages[:3]

In [ ]:
# ============================================================================
# EMBEDDINGS: OpenAI text-embedding-3-small
# ============================================================================
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# The vectorstore we'll be using
from langchain_chroma import Chroma

# The splitting and chunking strategy
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_core.documents import Document

docs = [Document(page_content=doc) for doc in passages]

In [ ]:
docs[:3]

In [ ]:
# ============================================================================
# SPLIT: 3,000-character chunks with 200 characters of overlap
# ============================================================================
splitter = RecursiveCharacterTextSplitter(chunk_size=3000,
                                          chunk_overlap=200)
chunked_docs = splitter.split_documents(docs)

In [ ]:
chunked_docs[:3]

### Create Vector DB and Retriever

The next cell builds `./wiki_db` only when the folder does not exist yet: re-running `from_documents` on an existing
folder would store every chunk a second time. Delete the folder to rebuild it.

In [ ]:
# ============================================================================
# VECTOR DB: Embed the Wikipedia chunks into a persistent Chroma collection
# ============================================================================
from pathlib import Path

# create vector DB of docs and embeddings - takes about 1 min
# from_documents on an existing folder would store every document a second time,
# so the vector DB is built only once - delete ./wiki_db to rebuild it
if Path("./wiki_db").exists():
    print("✅ ./wiki_db already exists - the next cell loads it (delete the folder to rebuild)")
else:
    chroma_db = Chroma.from_documents(documents=chunked_docs, collection_name='wiki_db',
                                      embedding=openai_embed_model,
                                      # need to set the distance function to cosine else it uses euclidean by default
                                      # check https://docs.trychroma.com/guides#changing-the-distance-function
                                      collection_metadata={"hnsw:space": "cosine"},
                                      persist_directory="./wiki_db")
    print(f"✅ Indexed {len(chunked_docs)} documents into ./wiki_db")

### Load Vector DB from disk

This works whether the previous cell just built the DB or it already existed.

In [ ]:
# load from disk
chroma_db = Chroma(persist_directory="./wiki_db",
                   collection_name='wiki_db',
                   embedding_function=openai_embed_model)

In [ ]:
chroma_db

The retriever returns up to 5 chunks whose cosine similarity to the query is at least 0.2.

In [ ]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity_score_threshold",
                                              search_kwargs={"k": 5, "score_threshold": 0.2})

---
## 🧭 Part 2: History-Aware Retriever

To maintain a seamless flow in conversations, especially in a Q&A setting, it's essential to incorporate historical interactions. Here’s how we achieve this:

### Defining a Sub-Chain for Historical Context

1. **Sub-Chain Creation:** We'll define a sub-chain that uses both historical messages and the latest user query. This sub-chain reformulates the question if it refers to any past interactions, ensuring the system, especially the vector database understands the context to return the most relevant documents to this newly reworded question.

2. **Using `MessagesPlaceholder`:** Our prompt construction involves a `MessagesPlaceholder` variable named `chat_history`. This setup allows us to input a list of messages using the `chat_history` key. The system integrates these messages, positioning them after its own responses and before the latest user question.

3. **Helper Function Usage:** We employ the `create_history_aware_retriever` function available [here](https://api.python.langchain.com/en/latest/chains/langchain.chains.history_aware_retriever.create_history_aware_retriever.html). This function is crucial for handling instances where the chat history might be empty and orchestrates the sequence of operations: `prompt | llm | StrOutputParser() | retriever`.

4. **Chain Construction:** The `create_history_aware_retriever` constructs a chain that processes inputs under the keys `input` and `chat_history`, ensuring the output schema aligns with that of a retriever.

By implementing these steps, our system can effectively utilize historical context to better understand and respond to user queries, thereby enhancing the conversational experience.

In [ ]:
# ============================================================================
# HISTORY-AWARE RETRIEVER: Rewrite follow-ups into standalone questions, then retrieve
# ============================================================================
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

######### REPHRASER ############
rephrase_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is."""

rephrase_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rephrase_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm, similarity_retriever, rephrase_prompt
)

---
## 🗃️ Part 3: Per-Session History in SQL

In many Q&A applications, the ability to engage in back-and-forth conversations with users is crucial. This necessitates the application having a form of "memory" to recall past interactions and apply this context to current queries.

However in most real-world conversational systems, multiple users or user sessions will be accessing the system simultaneously.

![](https://i.imgur.com/X4WivLu.gif)

Here we will show how you can use `SQLChatMessageHistory` such that we can store separate conversation histories per user or session which is often the need for real-world chatbots which will be accessed by many users at the same time. Instead of in-memory we can store it in a SQL database which can be used to store a lot of conversations.

We use a `get_session_history_db` function which is expected to take in a `session_id` and return a Message History object. Everything is stored in a SQL database. This `session_id` is used to distinguish between separate conversations, and should be passed in as part of the config when calling the new chain

We also use a `memory_buffer_window` function to only use the top-K last historical conversations before sending it to the LLM, basically our own implementation of `ConversationBufferWindowMemory`

In [ ]:
# ============================================================================
# SESSION STORE: One SQLChatMessageHistory per session_id, plus a last-K window
# ============================================================================
from pathlib import Path

from langchain_community.chat_message_histories import SQLChatMessageHistory

# every session's messages live in one SQLite file, keyed by session_id
SQLITE_URL = "sqlite:///memory.db"

# removes the memory database file so the demo starts from empty histories
# (comment this line out to keep the conversations of earlier runs)
Path(SQLITE_URL.removeprefix("sqlite:///")).unlink(missing_ok=True)

# used to retrieve conversation history from database
# based on a specific user or session ID
def get_session_history_db(session_id):
    history = SQLChatMessageHistory(session_id, SQLITE_URL)
    return history

# subset historical conversations based on last K conversation messages
# here by default we use the last 10 conversations (ai-human) as memory to the input prompt
def memory_buffer_window(messages, lastk_conversations=10):
    return messages[-(lastk_conversations*2):] # each conversation has 2 messages - (human prompt, AI response)

print('✅ Session store ready:', SQLITE_URL)

`memory_buffer_window` is plain list slicing, so it can be checked without any LLM call: with `lastk_conversations=2` it keeps the last 4 messages (2 human + 2 AI).

In [ ]:
# ============================================================================
# CHECK THE WINDOW: Keep the last 2 conversations (4 messages) of a 30-message list
# ============================================================================
print(memory_buffer_window(list(range(30)), lastk_conversations=2))

---
## 🔗 Part 4: Wire the History into the Chain

The QA chain trims the loaded history with `memory_buffer_window` before it reaches the prompt.
`RunnableWithMessageHistory` wraps the whole retrieval chain: on every call it loads the history of the `session_id`
passed in the config into `chat_history`, and afterwards saves the new question (`input`) and `answer` to that session.

In [ ]:
# ============================================================================
# MULTI-USER RAG CHAIN: windowed QA chain + RunnableWithMessageHistory
# ============================================================================
from langchain_classic.chains import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory

######### MULTI_USER RAG RESPONSE GENERATOR ############
qa_system_prompt = """You are an assistant for question-answering tasks.
                      Analyze the user question carefully and use the following pieces of
                      retrieved context to answer the question.
                      If the answer is not present in the context, just say that you don't know.
                      Keep the answer to the point.

                      Context:
                      {context}
                  """

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", """Question:
                     {input}

                     Answer:
                  """),
    ]
)

# custom RAG chain which looks at last K conversational messages
question_answer_chain = (
    RunnablePassthrough.assign(chat_history=lambda x: memory_buffer_window(x["chat_history"]))
      |
    qa_prompt
      |
    llm
      |
    StrOutputParser()
)
qa_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

############ CONVERSATIONAL RAG CHAIN ####################
conversational_rag_chain = RunnableWithMessageHistory(
    qa_rag_chain,
    get_session_history_db,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

---
## 👥 Part 5: Serve Several Users

`conv_rag_chatbot` sends one message for one session and prints the answer and its source documents. Two users, `bond007` and `jim003`, take turns below.

In [ ]:
# ============================================================================
# CHATBOT HELPER: One turn for one session
# ============================================================================
from IPython.display import display, Markdown

def conv_rag_chatbot(usersession_id, prompt):
    response = conversational_rag_chain.invoke(
                                {"input": prompt},
                                config={
                                    "configurable": {"session_id": usersession_id}
                                }
    )
    print('Answer:')
    display(Markdown(response['answer']))
    print('Sources:')
    for document in response['context']:
        print(document)
        print()

    return response

In [ ]:
us_id = 'bond007'
r = conv_rag_chatbot(us_id, 'What is the capital of India?')

In [ ]:
r = conv_rag_chatbot(us_id, 'Tell me more about it')

In [ ]:
us_id = 'jim003'
r = conv_rag_chatbot(us_id, 'What is the fastest animal on land?')

In [ ]:
us_id = 'bond007'
r = conv_rag_chatbot(us_id, 'Tell me about wildlife in India')

In [ ]:
us_id = 'jim003'
r = conv_rag_chatbot(us_id, 'tell me more about its different species')

Check the two follow-ups ("Tell me more about it", "tell me more about its different species"): each should be
resolved against its own user's earlier messages (India for `bond007`, the fastest land animal for `jim003`), even
though the two conversations are interleaved. The next cell reads the SQL store directly to confirm that each session
holds only its own turns: 6 messages for `bond007` (3 questions + 3 answers) and 4 for `jim003`.

In [ ]:
# ============================================================================
# CHECK ISOLATION: Read each session's stored messages from memory.db
# ============================================================================
for session_id in ["bond007", "jim003"]:
    messages = get_session_history_db(session_id).messages
    print(f"{session_id}: {len(messages)} messages")
    for message in messages:
        print(f"  {message.type:>5}: {message.content[:80]!r}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Build the Knowledge Base
- The Wikipedia subset and Chroma retriever are rebuilt here so the notebook runs on its own.

### 2. History-Aware Retriever
- Follow-up questions are rewritten into standalone questions before retrieval.

### 3. Per-Session History in SQL
- `SQLChatMessageHistory(session_id, "sqlite:///memory.db")` keeps every session's messages in one table, keyed by session.
- `memory_buffer_window` caps the prompt at the last K conversations.

### 4. Wire the History into the Chain
- `RunnableWithMessageHistory` loads and saves history around every call; the caller passes only `session_id`.

### 5. Serve Several Users
- Interleaved conversations stay separate because each turn reads and writes only its own session's history.

### Next Steps
- **`05_multimodal_rag_with_gpt_4o.ipynb`**: moves from text-only RAG to documents with tables and images.